# Importation de packages

In [ ]:
from utils import load_filosofi, load_bpe

# Données

Nous importons ici les données via une api (pour les données BPE) et via l'url pour les données filosofi en utilisant les fonctions load_bpe et load_filosofi écrites dans utils.py .
Il est à noter que pour éviter de retélécharger à chaque fois, ces bases sont sauvegardées en local après le premier chargement dans un dossier data qui se cree automatiquement.

In [ ]:
df_bpe = load_bpe(
    base_url="https://data.smartidf.services/api/explore/v2.1/catalog/datasets/bpe-2018/records",
    limit=100,
    data_dir="data",
    file_name="bpe.csv",
    force_reload=False
)

In [ ]:
df_filo = load_filosofi(
    url="https://www.insee.fr/fr/statistiques/fichier/5055909/BASE_TD_FILO_DEC_IRIS_2018.xlsx",
    header_row=4,
    data_dir="data",
    file_name="filosofi.csv",
    force_reload=False
)

Données géospatiales de communes pour l'estimation des distances aux équipements les plus proches.

In [ ]:
import geopandas as gpd

# 1. charger données
communes = gpd.read_file(
    "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions/bretagne/communes-bretagne.geojson"
)

# 2. reprojection en mètres (IMPORTANT)
communes = communes.to_crs(epsg=2154)

# 3. centroid (maintenant correct)
communes["centroid"] = communes.geometry.centroid

# 4. revenir en lat/lon pour OSM
communes_latlon = communes.set_geometry("centroid").to_crs(epsg=4326)

communes["lon"] = communes_latlon.geometry.x
communes["lat"] = communes_latlon.geometry.y

# 5. clean
communes = communes[["code", "nom", "lat", "lon", "geometry"]]

In [ ]:
communes.head()

Données du réseau routier OSM

In [ ]:
import osmnx as ox

place = "Bretagne, France"

G = ox.graph_from_place(place, network_type="drive")

# ajouter vitesse et temps
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

## Exploration de données

## Aperçu des données

In [ ]:
df_bpe.head()

In [ ]:
df_filo.head()

In [ ]:
print("Variables de bpe:", df_bpe.columns)
print("Variables de filosofi:",df_filo.columns)

Nous avons choisi de restreindre l’analyse à la région Bretagne afin de garantir un bon compromis entre précision géographique et faisabilité computationnelle.
L’utilisation du réseau routier OpenStreetMap nous permettra d'avoir une estimation des temps d’accès réalistes aux soins, ce qui constitue une amélioration significative par rapport aux approches basées uniquement sur la densité d’équipements.